In [4]:
import re
import pandas as pd

# =====================================================
# 1. PATHS
# =====================================================
SCOPUS_PATH = "downloads/scopus_export_Mar 30-2026_31c9e799-c6ca-4a69-bbd8-3bd2a75ca1fa.txt"
IEEE_PATH   = "downloads/IEEE Xplore Citation Plain Text Download 2026.3.30.19.20.42.txt"
GS_PATH     = "downloads/google_scholar.txt"

# =====================================================
# 2. LOAD FILES
# =====================================================
def load_file(path):
    try:
        return open(path, encoding="utf-8", errors="ignore").read()
    except FileNotFoundError:
        print(f"File not found: {path}")
        return ""

scopus = load_file(SCOPUS_PATH)
ieee   = load_file(IEEE_PATH)
gs     = load_file(GS_PATH)

# =====================================================
# 3. SCOPUS PARSER
# =====================================================
def parse_scopus_records(text):
    lines = text.splitlines()
    doc_idxs = [i for i, l in enumerate(lines) if l.strip().startswith("DOCUMENT TYPE:")]

    recs = []
    for idx, start in enumerate(doc_idxs):
        end = doc_idxs[idx + 1] if idx + 1 < len(doc_idxs) else len(lines)

        year_idx = None
        for i in range(start - 1, max(-1, start - 300), -1):
            if re.match(r"^\(\d{4}\)", lines[i].strip()):
                year_idx = i
                break

        title = ""
        if year_idx is not None:
            j = year_idx - 1
            while j >= 0 and not lines[j].strip():
                j -= 1
            if j >= 0:
                title = lines[j].strip()

        doi = ""
        for i in range(start, end):
            if lines[i].strip().startswith("DOI:"):
                doi = lines[i].split("DOI:")[1].strip()
                break

        recs.append({
            "Database": "Scopus",
            "Title": title,
            "DOI": doi,
            "Source": "Scopus"
        })

    return recs

# =====================================================
# 4. IEEE PARSER
# =====================================================
def parse_ieee_records(text):
    recs = []
    blocks = [b.strip() for b in re.split(r"\n\s*\n", text) if b.strip()]

    for b in blocks:
        m_title = re.search(r"\"([^\"]+)\"", b)
        if not m_title:
            continue

        title = m_title.group(1).strip()

        m_doi = re.search(r"\bdoi:\s*([0-9]+\.[0-9]+\/[^\s,;]+)", b, flags=re.I)
        doi = m_doi.group(1) if m_doi else ""

        recs.append({
            "Database": "IEEE",
            "Title": title,
            "DOI": doi,
            "Source": "IEEE"
        })

    return recs

# =====================================================
# 5. GOOGLE SCHOLAR PARSER
# =====================================================
def parse_gs_records(text):
    recs = []

    blocks = re.split(r"\n\s*\[\d+\]\s*", text)

    for b in blocks:
        if "Title:" not in b:
            continue

        title_match = re.search(r"Title:\s*(.*)", b)
        title = title_match.group(1).strip() if title_match else ""

        recs.append({
            "Database": "Google Scholar",
            "Title": title,
            "DOI": "",
            "Source": "Google Scholar"
        })

    return recs

# =====================================================
# 6. RUN PARSING
# =====================================================
scopus_recs = parse_scopus_records(scopus)
ieee_recs   = parse_ieee_records(ieee)
gs_recs     = parse_gs_records(gs)

print("======== COUNTS ========")
print("Scopus:", len(scopus_recs))
print("IEEE:", len(ieee_recs))
print("Google Scholar:", len(gs_recs))
print("TOTAL:", len(scopus_recs) + len(ieee_recs) + len(gs_recs))

# =====================================================
# 7. MERGE DATA
# =====================================================
df = pd.DataFrame(scopus_recs + ieee_recs + gs_recs)

# =====================================================
# 8. CLEAN TITLES (IMPORTANT FOR DUPLICATES)
# =====================================================
df["Title_clean"] = (
    df["Title"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# =====================================================
# 9. ADD PROJECT COLUMNS
# =====================================================
df.insert(0, "Record ID", [f"R{str(i+1).zfill(4)}" for i in range(len(df))])

df["Domain"] = ""
df["Task"] = ""
df["Modality"] = ""
df["Fusion_Type"] = ""
df["Application"] = ""

df["Decision"] = ""
df["Reason"] = ""
df["Score"] = ""

# =====================================================
# 10. SAVE FILES
# =====================================================
CSV_OUT = "articles_raw.csv"
EXCEL_OUT = "articles_raw.xlsx"

df.to_csv(CSV_OUT, index=False)
df.to_excel(EXCEL_OUT, index=False)

print("Saved CSV:", CSV_OUT)
print("Saved Excel:", EXCEL_OUT)

======== COUNTS ========
Scopus: 89
IEEE: 5
Google Scholar: 30
TOTAL: 124


ModuleNotFoundError: No module named 'openpyxl'